In [0]:

# ============================================================
# CELDA 1. INTEGRACIÓN DE VUELOS Y PRONÓSTICOS METEOROLÓGICOS
# ============================================================

# Objetivo:
# Relacionar los vuelos con el pronóstico meteorológico
# disponible desde el día anterior.
#
# La unión se realiza usando:
# 1. Aeropuerto de origen.
# 2. Hora programada de salida convertida a UTC.
#
# Los datos meteorológicos provienen de Open-Meteo
# y corresponden al pronóstico previous_day1.

from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

# ------------------------------------------------------------
# 1. CARGAR LAS TABLAS
# ------------------------------------------------------------

vuelos = spark.table(
    "workspace.gold_flights.flights_features"
)

clima = spark.table(
    "workspace.silver_flights.weather_clean"
)

print("Tablas cargadas correctamente.")

# ------------------------------------------------------------
# 2. PREPARAR LOS PRONÓSTICOS
# ------------------------------------------------------------

# Usamos únicamente las variables meteorológicas
# que estaban disponibles antes del vuelo.

clima_features = clima.select(
    F.col("airport").alias("weather_airport"),
    F.col("weather_time_utc").alias("forecast_hour_utc"),
    F.col("temperature_2m"),
    F.col("precipitation"),
    F.col("wind_speed_10m"),
    F.col("relative_humidity_2m"),
    F.when(
        F.col("cloud_cover") < 0, 0
    ).when(
        F.col("cloud_cover") > 100, 100
    ).otherwise(
        F.col("cloud_cover")
    ).alias("cloud_cover"),
    F.col("pressure_msl"),
    F.col("forecast_lead_hours")
)

# ------------------------------------------------------------
# 3. UNIR LOS VUELOS CON EL CLIMA
# ------------------------------------------------------------

# LEFT JOIN conserva todos los vuelos.
# Si no hay clima para un horario específico,
# las variables meteorológicas quedan como NULL.

df_weather_gold = vuelos.join(
    clima_features,
    (vuelos["origin"] == clima_features["weather_airport"]) &
    (
        vuelos["weather_hour_utc"] ==
        clima_features["forecast_hour_utc"]
    ),
    how="left"
)

# Eliminar columnas auxiliares duplicadas
df_weather_gold = df_weather_gold.drop(
    "weather_airport",
    "forecast_hour_utc"
)

# ------------------------------------------------------------
# 4. VERIFICAR LOS RESULTADOS
# ------------------------------------------------------------

total_vuelos = vuelos.count()
total_integrados = df_weather_gold.count()

con_clima = df_weather_gold.filter(
    F.col("temperature_2m").isNotNull()
).count()

sin_clima = total_integrados - con_clima

porcentaje = round(
    100 * con_clima / total_integrados, 2
) if total_integrados > 0 else 0

print("=" * 55)
print("INTEGRACIÓN GOLD: VUELOS + CLIMA")
print("=" * 55)

print("Vuelos originales:", total_vuelos)
print("Registros después del JOIN:", total_integrados)
print("Vuelos con pronóstico:", con_clima)
print("Vuelos sin pronóstico:", sin_clima)
print("Cobertura meteorológica:", porcentaje, "%")

if total_integrados != total_vuelos:
    print(
        "ADVERTENCIA: el JOIN modificó el número de registros."
    )
else:
    print("VALIDACIÓN: se conservó el número de vuelos.")

display(
    df_weather_gold.select(
        "flight_date",
        "origin",
        "dest",
        "airline",
        "scheduled_departure_utc",
        "weather_hour_utc",
        "temperature_2m",
        "precipitation",
        "wind_speed_10m",
        "cloud_cover",
        "delayed_15"
    ).limit(10)
)

print("INTEGRACIÓN GOLD COMPLETADA.")


In [0]:

# ============================================================
# CELDA 2. GUARDAR GOLD CON VARIABLES METEOROLÓGICAS
# ============================================================

# Conservamos los vuelos con pronóstico disponible.
# Los vuelos sin clima no se eliminan de la tabla
# operativa original; solamente se excluyen de esta
# tabla destinada a comparar los dos modelos.

from pyspark.sql import functions as F

TABLA_DESTINO = "workspace.gold_flights.flights_weather_features"

# Excluimos solamente los vuelos sin información meteorológica.
df_modelos = df_weather_gold.filter(
    F.col("temperature_2m").isNotNull()
)

# Validamos que las seis variables meteorológicas estén completas.
variables_clima = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

faltantes = df_modelos.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in variables_clima
]).first().asDict()

print("Valores faltantes por variable:", faltantes)

if any(valor > 0 for valor in faltantes.values()):
    raise ValueError(
        "Hay variables meteorológicas incompletas. Revisar antes de guardar."
    )

# Crear el esquema Gold, si todavía no existe.
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS workspace.gold_flights"
)

# Guardar en Delta.
(
    df_modelos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_DESTINO)
)

print("=" * 55)
print("TABLA GOLD CON CLIMA GUARDADA CORRECTAMENTE")
print("Tabla:", TABLA_DESTINO)
print("Vuelos guardados:", df_modelos.count())
print("=" * 55)

display(
    spark.table(TABLA_DESTINO)
    .select(
        "flight_date",
        "origin",
        "dest",
        "weather_hour_utc",
        "temperature_2m",
        "precipitation",
        "wind_speed_10m",
        "cloud_cover",
        "delayed_15"
    )
    .limit(10)
)
